In [1]:
#CREACION DE CARPETAS
!mkdir -p datos/crudos
!mkdir -p datos/procesados
!mkdir -p datos/externos
print("Carpetas creadas !!!!!!")

Carpetas creadas !!!!!!


In [2]:
#IMPORTACION DE LIBRERIAS
import pandas as pd #Manejar datos como excel
import sqlite3 #Para base de datos ligera
import json #Abrir y manipular archivos json
from pathlib import Path #Atajos
from datetime import time #Convertir horas
print("Librerias importadas")

Librerias importadas


In [3]:
#Rutas de trabajo ya definidas "atajos"
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR/"datos"/"crudos"
DATA_EXTERNAL = BASE_DIR/"datos"/"externos"
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
print(f"Datos crudos-> {DATA_RAW}")
print(f"Datos procesados-> {DATA_PROCESSED}")
print(f"Datos externos -> {DATA_EXTERNAL}")

Datos crudos-> /content/datos/crudos
Datos procesados-> /content/datos/procesados
Datos externos -> /content/datos/externos


In [ ]:
#CREAR CONEXION
conexion = sqlite3.connect(DATA_RAW/"zomato_delivery.sqlite")
cursor = conexion.cursor()
print("Base de datos conectada")

In [ ]:
#VISUALIZANDO LA BASE DE DATOS
print("Productos en la base de datos")
df_delivery_sql = pd.read_sql_query("SELECT * FROM deliveries;", conexion)
df_delivery_sql

In [ ]:
#ABRIENDO ARCHIVO CSV
ruta_csv = DATA_RAW/"zomato_delivery.csv"
df_delivery_csv = pd.read_csv(ruta_csv)
print("Leyendo el archivo csv\n")
df_delivery_csv

In [ ]:
#ABRIENDO ARCHIVO CSV
ruta_csv_sed = DATA_RAW/"zomato_delivery_sed.csv"
df_delivery_sed_csv = pd.read_csv(ruta_csv_sed)
print("Leyendo el archivo csv\n")
df_delivery_sed_csv

In [ ]:
#ABRIENDO ARCHIVO JSON
ruta_json = DATA_RAW/"zomato_delivery.json"
df_delivery_json = pd.read_json(ruta_json)

print("Informacion geografica y de vehiculos")
df_delivery_json

In [ ]:
#INTEGRAR VENTAS CSV + SQL
df_integrado = pd.merge(df_delivery_csv, df_delivery_sql, on="ID", how="left")
print("\n Previsualización")
df_integrado

In [ ]:
#INTEGRAR DF_INTEGRADO + JSON
df_integrado = pd.merge(df_integrado, df_delivery_json, on="ID", how="left")
print("Dataset final")
df_integrado

In [ ]:
#INTEGRAR DF_INTEGRADO + SEMILLA
df_final = pd.concat([df_integrado, df_delivery_sed_csv], ignore_index=True)
df_final = df_final.drop_duplicates(subset=["ID"], keep="last")
print("Dataset final")
df_final

In [ ]:
#REENOMBRAMIENTO DE COLUMNAS
renombrar_columnas = {
    'ID': 'ID',
    'Delivery_person_ID': 'ID_Repartidor',
    'Delivery_person_Age': 'Edad_Repartidor',
    'Delivery_person_Ratings': 'Calificacion_Repartidor',
    'Order_Date': 'Fecha_Pedido',
    'Time_Orderd': 'Hora_Pedido',
    'Time_Order_picked': 'Hora_Recogida_Pedido',
    'Weather_conditions': 'Condiciones_Climaticas',
    'Road_traffic_density': 'Densidad_Trafico',
    'Type_of_order': 'Tipo_Pedido',
    'Festival': 'Festival',
    'City': 'Ciudad',
    'Time_taken (min)': 'Tiempo_Entrega_min',
    'Restaurant_latitude': 'Latitud_Restaurante',
    'Restaurant_longitude': 'Longitud_Restaurante',
    'Delivery_location_latitude': 'Latitud_Ubicacion_Entrega',
    'Delivery_location_longitude': 'Longitud_Ubicacion_Entrega',
    'Vehicle_condition': 'Condicion_Vehiculo',
    'Type_of_vehicle': 'Tipo_Vehiculo',
    'multiple_deliveries': 'Multiples_Entregas'
}

df_final = df_final.rename(columns=renombrar_columnas)

print("Nombres de columnas traducidos:")
print(df_final.columns.tolist())

In [ ]:
# GUARDAR DATASET FINAL
ruta_dataset = DATA_PROCESSED / "dataset_integrado.csv"
df_final.to_csv(ruta_dataset, index=False)
print(f"Dataset guardado en: {ruta_dataset}")